In [1]:
import ee
import json
import re
import os
import glob
ee.Authenticate(auth_mode="gcloud")
ee.Initialize(project= 'rmrs-wildfire-treatments')


In [2]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=kDVlg2DgYkRcZYyoDfEMGvOK7UL6Uq&access_type=offline&code_challenge=6g02peUeW_1C7V7E53fuWBYIpgLdOQp1OsF8--W7vm8&code_challenge_method=S256


You are now logged in as [hannah.vandusen@usda.gov].
Your current project is [rmrs-wildfire-treatments].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [4]:
!earthengine -h

usage: earthengine [-h] [--ee_config EE_CONFIG]
                   [--service_account_file SERVICE_ACCOUNT_FILE]
                   [--project PROJECT_OVERRIDE]
                   {authenticate,acl,asset,cp,create,ls,alpha,du,mv,model,rm,set_project,task,unset_project,upload,upload_manifest,upload_table_manifest}
                   ...

Earth Engine Command Line Interface.

options:
  -h, --help            show this help message and exit
  --ee_config EE_CONFIG
                        Path to the earthengine configuration file. Defaults
                        to "~/.config\earthengine\credentials".
  --service_account_file SERVICE_ACCOUNT_FILE
                        Path to a service account credentialsfile. Overrides
                        any ee_config if specified.
  --project PROJECT_OVERRIDE
                        Specifies a Google Cloud Platform Project id to
                        override the call.

Commands:
  {authenticate,acl,asset,cp,create,ls,alpha,du,mv,model,rm,set

In [5]:
!earthengine set_project rmrs-wildfire-treatments

Successfully saved project id


In [6]:
# import user-defined settings

# get pathfile of this script
#!pip install ipynbname
import ipynbname
notebook_path = ipynbname.path()
project_root = notebook_path.parent

import pandas as pd
import ast

# Full local path to user-input csv file
input_path = os.path.join(project_root, "user_input/user_input_general.csv")

# read user-input CSV
df = pd.read_csv(input_path)

# container for created objects from user input csv
user_inputs = {}

for _, row in df.iterrows():
    # Skip rows flagged as R expressions 
    if row["treat_as_R_expression"]: 
        continue
    
    raw = row["value"]

    # Try safe literal parsing; fall back to raw string 
    try: 
        val = ast.literal_eval(raw) 
    except Exception: 
        val = raw 
        
    user_inputs[row["name"]] = val



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
user_inputs["processed_data_directory"]

'C:/Users/HannahVanDusen/Box/External Wildfire Treatment Outcomes/klamath/data/klamath'

In [12]:
# upload day of burn rasters to GEE bucket

# Set your local directory here:
local_directory = user_inputs["processed_data_directory"]

# Subfolder to upload
subfolder = "date_of_burn/dob_tif/"

# Full local path to upload
local_path = os.path.join(local_directory, subfolder)

# Destination in bucket
bucket_folder = "gs://bb-gee-bucket/klamath/dob_parks/" # if changing this, have to go to https://console.cloud.google.com/storage/browser/bb-gee-bucket/ and add new folder

# List all .tif files, but exclude those ending with _tmp.tif
tif_files = [f for f in glob.glob(os.path.join(local_path, "*.tif")) if not f.endswith("_tmp.tif")]

for tif in tif_files:
    !gsutil cp "{tif}" "{bucket_folder}"

Copying file://C:\Users\HannahVanDusen\Box\External Wildfire Treatment Outcomes\klamath\data\klamath\date_of_burn\dob_tif\CA3966012280920200817_1_dob.tif [Content-Type=image/tiff]...
/ [0 files][    0.0 B/  7.8 MiB]                                                
/ [0 files][320.0 KiB/  7.8 MiB]                                                
-
\
\ [0 files][  4.1 MiB/  7.8 MiB]                                                
|
/
/ [0 files][  7.8 MiB/  7.8 MiB]                                                
/ [1 files][  7.8 MiB/  7.8 MiB]                                                

Operation completed over 1 objects/7.8 MiB.                                      
Copying file://C:\Users\HannahVanDusen\Box\External Wildfire Treatment Outcomes\klamath\data\klamath\date_of_burn\dob_tif\CA4018412336320170808_1_dob.tif [Content-Type=image/tiff]...
/ [0 files][    0.0 B/ 23.1 KiB]                                                
/ [1 files][ 23.1 KiB/ 23.1 KiB]                         

In [23]:
# Specify the bucket where the source images are stored.
GCS_BUCKET = 'gs://bb-gee-bucket/klamath/dob_parks'

# Specify the new asset's path (ensure you have project write permission).
ASSET_FOLDER = 'projects/rmrs-wildfire-treatments/assets/klamath/dob_parks/'

# List the contents of the cloud folder.
cloud_files = !gsutil ls {GCS_BUCKET + '/*.tif'}

# Upload each raster separately
for file in cloud_files:
    # Extract filename without path
    filename = os.path.basename(file).replace(".tif", "")
    
    # Extract metric and year from filename (assumes structure e.g.{MTBS_ID}_1_dob.tif)
    match = re.match(r"([A-Z]{2}\d{19})_1_dob", filename)
    if not match:
        print(f"Skipping {filename}: Does not match expected pattern")
        continue

    # metric, year = match.groups()
    # band_id = f"{metric}_{year}"

    # Define asset name
    asset_name = f"{ASSET_FOLDER}{filename}"
    band_id="b1"

    # Delete existing asset 
    try: 
        ee.data.deleteAsset(asset_name) 
        print(f"Deleted existing asset: {asset_name}") 
    except Exception: 
        pass

    # Define the asset properties for upload
    asset = {
        'name': asset_name,
        'tilesets': [{'sources': [{'uris': [file]}]}],
        'bands': [{'id': band_id, 'tilesetBandIndex': 0}]
    }

    # Start ingestion
    task_id = ee.data.newTaskId()[0]
    # print(f"Uploading {filename} as {asset_name} with band {band_id}")
    print(f"Uploading {filename}")
    ee.data.startIngestion(task_id, asset)

Uploading CA3966012280920200817_1_dob
Uploading CA4018412336320170808_1_dob
Uploading CA4022812303620170913_1_dob
Uploading CA4034312338320150731_1_dob
Uploading CA4035012303620210730_1_dob
Uploading CA4035712344220150731_1_dob
Uploading CA4035812356120150806_1_dob
Uploading CA4038712317320150730_1_dob
Uploading CA4045612355620150806_1_dob
Uploading CA4046512305420150731_1_dob
Uploading CA4049212319820150803_1_dob
Uploading CA4054612309020120905_1_dob
Uploading CA4060412308120150731_1_dob
Uploading CA4064212358620150731_1_dob
Uploading CA4065012263020180723_1_dob
Uploading CA4068012335620150731_1_dob
Uploading CA4069112352420150610_1_dob
Uploading CA4075212333720210731_1_dob
Uploading CA4077112312420170831_1_dob
Uploading CA4079212335020120711_1_dob
Uploading CA4091312343720150731_1_dob
Uploading CA4091612363420210829_1_dob
Uploading CA4094312242720180905_1_dob
Uploading CA4103512348820130810_1_dob
Uploading CA4114112280520140802_1_dob
Uploading CA4114212301620210731_1_dob
Uploading CA

In [24]:
ee.data.startIngestion(ee.data.newTaskId()[0], asset)

{'id': 'LW4YOLRCO24KTS5CGRHTZRO7',
 'name': 'projects/rmrs-wildfire-treatments/operations/LW4YOLRCO24KTS5CGRHTZRO7',
 'started': 'OK'}

In [22]:
!earthengine task list

CSCFRB6SJEDF27PIV3JZJF3B  Upload        Ingest image: "projects/rmrs-wildfire-tr..  FAILED     Asset 'projects/rmrs-wildfire-treatments/assets/klamath/DOB_parks' does not exist or doesn't allow this operation.
4SUIJTJVMV52NESN7P4XTI67  Upload        Ingest image: "projects/rmrs-wildfire-tr..  FAILED     Asset 'projects/rmrs-wildfire-treatments/assets/klamath/DOB_parks' does not exist or doesn't allow this operation.
U55DPBD6RQNPSMMRINF7JYYC  Upload        Ingest image: "projects/rmrs-wildfire-tr..  FAILED     Asset 'projects/rmrs-wildfire-treatments/assets/klamath/DOB_parks' does not exist or doesn't allow this operation.
WL7OIN5PEYHSVUNTR7JQLCF5  Upload        Ingest image: "projects/rmrs-wildfire-tr..  FAILED     Asset 'projects/rmrs-wildfire-treatments/assets/klamath/DOB_parks' does not exist or doesn't allow this operation.
SFD6ELDB5HBDJVEFVVK3NUVI  Upload        Ingest image: "projects/rmrs-wildfire-tr..  FAILED     Asset 'projects/rmrs-wildfire-treatments/assets/klamath/DOB_parks